# MVP2 Pingpong — PYNQ Test Notebook

Tests the full pipeline: CORDIC source → FDTD solver (64×64) → field magnitude → ping-pong BRAMs → PS readback.

**Run cells top to bottom. Each cell prints a PASS/FAIL result.**

---
## Cell 1 — Load bitstream

In [ ]:
from pynq import Bitstream
import time

bs = Bitstream('/home/xilinx/mvp2_pingpong.bit')
bs.download()
print('✓ Bitstream loaded')

---
## Cell 2 — AXI GPIO helpers

In [ ]:
from pynq import MMIO
import numpy as np

class GPIO:
    """Dual-channel AXI GPIO wrapper."""
    def __init__(self, base):
        self.m = MMIO(base, 0x10000)
    def ch1_w(self, v):  self.m.write(0x00, int(v) & 0xFFFFFFFF)
    def ch2_w(self, v):  self.m.write(0x08, int(v) & 0xFFFFFFFF)
    def ch1_r(self):     return self.m.read(0x00)
    def ch2_r(self):     return self.m.read(0x08)

ctrl   = GPIO(0x41200000)  # control  PS→PL
stat   = GPIO(0x41210000)  # status   PL→PS
smag_a = GPIO(0x41220000)  # smag_a BRAM
smag_b = GPIO(0x41230000)  # smag_b BRAM

def probe_cell(row, col):
    """Read |E| magnitude at grid cell (row, col) from the completed buffer."""
    sel  = (stat.ch2_r() >> 5) & 1   # pp_read_sel: 0=smag_a, 1=smag_b
    gpio = smag_a if sel == 0 else smag_b
    gpio.ch1_w((1 << 12) | (row * 64 + col))
    time.sleep(1e-5)
    return gpio.ch2_r() & 0xFFFF

def read_frame():
    """Read the full 64×64 magnitude frame from the completed ping-pong buffer."""
    sel  = (stat.ch2_r() >> 5) & 1
    gpio = smag_a if sel == 0 else smag_b
    frame = np.zeros(4096, dtype=np.uint16)
    for addr in range(4096):
        gpio.ch1_w((1 << 12) | addr)
        frame[addr] = gpio.ch2_r() & 0xFFFF
    return frame.reshape(64, 64)

def read_status():
    s = stat.ch2_r()
    return {
        'solver_done':    (s >> 0) & 1,
        'source_valid':   (s >> 1) & 1,
        'mag_done':       (s >> 2) & 1,
        'mag_busy':       (s >> 3) & 1,
        'source_latched': (s >> 4) & 1,
        'pp_read_sel':    (s >> 5) & 1,
        'pp_frame_ready': (s >> 6) & 1,
        'source_q313':    (s >> 7) & 0xFFFF,
        'checksum':       stat.ch1_r(),
    }

print('✓ GPIO helpers ready')

---
## Cell 3 — Test 1: Solver is running (checksum changes)

In [ ]:
# Enable solver with source at grid centre
PHASE_STEP  = 200
AMPLITUDE   = 4096   # 0.5 in Q3.13
SOURCE_ROW  = 32
SOURCE_COL  = 32
SOURCE_ADDR = SOURCE_ROW * 64 + SOURCE_COL

ctrl.ch1_w((AMPLITUDE << 16) | PHASE_STEP)
ctrl.ch2_w((1 << 12) | SOURCE_ADDR)   # solver_enable=1

time.sleep(0.1)

checksums = [stat.ch1_r() for _ in range(5) if not time.sleep(0.02)]
unique = len(set(checksums))

print(f"Checksums: {[hex(c) for c in checksums]}")
if unique >= 3:
    print(f'PASS — solver is running ({unique}/5 unique checksum values)')
else:
    print(f'FAIL — checksum not changing (got {unique} unique values); check solver_enable')

---
## Cell 4 — Test 2: Source cell has energy

In [ ]:
time.sleep(0.1)
samples = [probe_cell(SOURCE_ROW, SOURCE_COL) for _ in range(6) if not time.sleep(0.05)]

print(f"|E| at source ({SOURCE_ROW},{SOURCE_COL}): {samples}")
if max(samples) > 100:
    print(f'PASS — source cell has energy (max={max(samples)})')
else:
    print(f'FAIL — source cell reads near zero; check amplitude and source_addr')

---
## Cell 5 — Test 3: Wave propagates outward (near cell > far cell > edge)

In [ ]:
time.sleep(0.2)

src  = probe_cell(SOURCE_ROW, SOURCE_COL)          # injection point
near = probe_cell(SOURCE_ROW, SOURCE_COL + 8)      # 8 cells away
far  = probe_cell(SOURCE_ROW, SOURCE_COL + 20)     # 20 cells away
edge = probe_cell(1, 1)                            # near corner (PML should absorb)

print(f"source({SOURCE_ROW},{SOURCE_COL}) = {src}")
print(f"near  ({SOURCE_ROW},{SOURCE_COL+8}) = {near}")
print(f"far   ({SOURCE_ROW},{SOURCE_COL+20}) = {far}")
print(f"edge  (1,1)        = {edge}")
print()

passes = []
if src > 50:      passes.append(f'source has energy ({src})')
else:             passes.append(f'FAIL: source too low ({src})')
if near > 0:      passes.append(f'wave reached near cell ({near})')
else:             passes.append(f'FAIL: near cell = 0')
if edge < src//4: passes.append(f'PML absorbed edge ({edge} << {src})')
else:             passes.append(f'WARN: edge not well absorbed (edge={edge}, src={src})')

for p in passes:
    print(('PASS' if 'FAIL' not in p and 'WARN' not in p else p.split(':')[0]) + ' — ' + p)

---
## Cell 6 — Test 4: Ping-pong buffer swaps (frame_ready pulses)

In [ ]:
sel_values = []
for _ in range(10):
    sel_values.append((stat.ch2_r() >> 5) & 1)
    time.sleep(0.05)

print(f"pp_read_sel over time: {sel_values}")
if 0 in sel_values and 1 in sel_values:
    print('PASS — ping-pong is swapping buffers')
else:
    print('WARN — pp_read_sel did not toggle; ping-pong may need more time')

---
## Cell 7 — Test 5: Change source location, verify field moves

In [ ]:
def measure_energy_at(row, col, settle=0.3):
    ctrl.ch2_w((1 << 12) | (row * 64 + col))
    time.sleep(settle)
    return max(probe_cell(row, col) for _ in range(3))

# Source at top-left quadrant
e_tl = measure_energy_at(16, 16)
# Source at bottom-right quadrant  
e_br = measure_energy_at(48, 48)
# Restore centre
ctrl.ch2_w((1 << 12) | SOURCE_ADDR)

print(f"Source at (16,16) → |E| at (16,16) = {e_tl}")
print(f"Source at (48,48) → |E| at (48,48) = {e_br}")
print()
if e_tl > 50 and e_br > 50:
    print('PASS — source injection address is configurable at runtime')
else:
    print('FAIL — one or both source locations read zero')

---
## Cell 8 — Test 6: Frequency control (phase_step changes oscillation rate)

In [ ]:
ctrl.ch2_w((1 << 12) | SOURCE_ADDR)  # source back at centre

def checksum_change_rate(phase_step, n=8, delay=0.02):
    """How many of n consecutive checksums are unique (proxy for solver activity)."""
    ctrl.ch1_w((AMPLITUDE << 16) | phase_step)
    time.sleep(0.1)
    cs = [stat.ch1_r() for _ in range(n) if not time.sleep(delay)]
    return len(set(cs))

slow = checksum_change_rate(50)
fast = checksum_change_rate(800)
ctrl.ch1_w((AMPLITUDE << 16) | PHASE_STEP)  # restore default

print(f"PHASE_STEP= 50 → {slow}/8 unique checksums")
print(f"PHASE_STEP=800 → {fast}/8 unique checksums")
print()
if slow >= 3 and fast >= 3:
    print('PASS — solver runs at both frequencies')
else:
    print('WARN — low solver activity at one setting; check solver_enable')

---
## Cell 9 — Read full 64×64 frame and display summary

In [ ]:
ctrl.ch1_w((AMPLITUDE << 16) | PHASE_STEP)
ctrl.ch2_w((1 << 12) | SOURCE_ADDR)
time.sleep(0.2)

print("Reading full 64×64 frame (4096 cells)...")
frame = read_frame()

nonzero = np.count_nonzero(frame)
print(f"Max  |E| = {frame.max()}")
print(f"Mean |E| = {frame.mean():.1f}")
print(f"Nonzero  = {nonzero} / 4096 cells ({100*nonzero/4096:.1f}%)")
print(f"Source cell ({SOURCE_ROW},{SOURCE_COL}) = {frame[SOURCE_ROW, SOURCE_COL]}")
print()

if frame.max() > 0 and nonzero > 10:
    print('PASS — full frame read successfully')
else:
    print('FAIL — frame appears empty')

---
## Cell 10 — Plot the 64×64 magnitude field (requires matplotlib)

In [ ]:
try:
    import matplotlib.pyplot as plt
    %matplotlib inline

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: magnitude heatmap
    im = axes[0].imshow(frame, cmap='hot', origin='lower', aspect='equal')
    axes[0].scatter([SOURCE_COL], [SOURCE_ROW], c='cyan', s=80, marker='x',
                    linewidths=2, label='source')
    axes[0].set_title('|E| magnitude — 64×64 FDTD grid')
    axes[0].set_xlabel('column'); axes[0].set_ylabel('row')
    axes[0].legend()
    plt.colorbar(im, ax=axes[0], label='|E| (Q3.13)')

    # Right: horizontal cross-section through source row
    axes[1].plot(frame[SOURCE_ROW, :], color='orangered', lw=2)
    axes[1].axvline(SOURCE_COL, color='cyan', linestyle='--', label='source')
    axes[1].set_title(f'|E| cross-section at row {SOURCE_ROW}')
    axes[1].set_xlabel('column'); axes[1].set_ylabel('|E| magnitude')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/home/xilinx/fdtd_result.png', dpi=150)
    plt.show()
    print('Plot saved to /home/xilinx/fdtd_result.png')

except ImportError:
    print('matplotlib not available — skipping plot')
    print('Frame row 32 (cross-section):')
    print(frame[SOURCE_ROW, :])

---
## Cell 11 — Summary

In [ ]:
s = read_status()
print("=== Final status ===")
for k, v in s.items():
    print(f"  {k:20s} = {hex(v) if k == 'checksum' else v}")

print()
print("=== Register map reminder ===")
print("  0x41200000  ctrl    CH1={amplitude,phase_step}  CH2={sample_req,mag_mode,solver_enable,source_addr}")
print("  0x41210000  status  CH1=solver_checksum  CH2={source_q313,pp_frame_ready,pp_read_sel,...}")
print("  0x41220000  smag_a  CH1={enb,addrb}  CH2={doutb}")
print("  0x41230000  smag_b  same")